# Gröbner Basis Verification for LSS on osp(1|2)
### Checking inconsistency of the graded left-symmetric identity over GF(3)

This notebook replicates and verifies the SymPy Gröbner basis computations from `linear_sys.py`,
using SageMath's more robust polynomial ideal infrastructure.

**Theoretical claim being verified:** The polynomial ideal generated by the graded left-symmetric
identity constraints, with Burde's structure constants included as indeterminates, equals the
whole ring over GF(3). Equivalently, the Gröbner basis is {1}, certifying that no solution
exists over any field of characteristic 3.

---
## Part 0 — Setup Check

Run this cell first to confirm SageMath is active (not plain Python).

In [33]:
# If this prints a Sage version string, your kernel is correctly set to SageMath.
# If it throws NameError, you are using a plain Python kernel — switch kernels first.
print(version())
print("Sage is active ✓")

SageMath version 10.7, Release Date: 2025-08-09
Sage is active ✓


/var/folders/db/kjkgjcbj5619ftndd0m2xsyr0000gn/T/ipykernel_32709/3462063063.py:3: DeprecationWarning: Use sage.version instead.
See https://github.com/sagemath/sage/issues/39015 for details.
  print(version())


---
## Part 1 — Define the Polynomial Ring

We work in the ring:

$$R = \mathbb{F}_3[\tau_1, \ldots, \tau_n, u, v] \quad \text{(Structure 1)}$$
$$R = \mathbb{F}_3[\tau_1, \ldots, \tau_m, g, g_{\mathrm{inv}}] \quad \text{(Structure 2)}$$

**Critical:** Both the unknown structure constants (the τᵢ or b-variables) AND Burde's
free parameters (u, v for Structure 1; g for Structure 2) are treated as **polynomial
indeterminates** — not as scalars. This is what makes the Gröbner basis result valid
over any field of characteristic 3, not just over GF(3) itself.

In [55]:
# ============================================================
# Choose which structure to verify: set to 1 or 2
# ============================================================
STRUCTURE = 3

# Base field: GF(3) = F_3, the prime field of characteristic 3
F3 = GF(3)
F9 = GF(9)

if STRUCTURE == 1:
    # Structure 1 parameters: u, v (and w = v - u is derived, not independent)
    # Unknown structure constants: b_X_Y_Z symbols
    # After Stage 1 linear reduction in SymPy, we are left with ~33 free tau parameters.
    # Replace the list below with the actual variable names output by your SymPy code.
    # The names here follow the tau_i naming from sp.linsolve.
    
    var_names = (
        # Burde's free parameters for Structure 1 (treat as indeterminates)
        ['u', 'v'] +
        # Free structure constant unknowns after Stage 1 linear reduction
        # REPLACE THIS LIST with the actual output of:
        #   print([str(s) for s in unknowns2])
        # from your SymPy code after running structure_version = 1
        ['b_Tm_E_Tm', 'b_Tm_E_Tp', 'b_Tm_F_Tm', 'b_Tm_F_Tp', 'b_Tm_H_Tm', 'b_Tm_H_Tp', 'b_Tm_Tp_E', 'b_Tm_Tp_F', 'b_Tm_Tp_H', 'b_Tp_E_Tm', 'b_Tp_E_Tp', 'b_Tp_F_Tm', 'b_Tp_F_Tp', 'b_Tp_H_Tm', 'b_Tp_H_Tp']  # placeholder: adjust count and names
    )
    
elif STRUCTURE == 2:
    # Structure 2 parameter: g (and ginv = g^{-1}, enforced by g*ginv - 1 = 0)
    var_names = (
        ['g', 'ginv'] +
        # REPLACE with actual unknowns2 names from structure_version = 2
        ['b_Tm_E_Tm', 'b_Tm_E_Tp', 'b_Tm_F_Tm', 'b_Tm_F_Tp', 'b_Tm_H_Tm', 'b_Tm_H_Tp', 'b_Tm_Tp_E', 'b_Tm_Tp_F', 'b_Tm_Tp_H', 'b_Tp_E_Tm', 'b_Tp_E_Tp', 'b_Tp_F_Tm', 'b_Tp_F_Tp', 'b_Tp_H_Tm', 'b_Tp_H_Tp']  # placeholder: adjust count and names
    )
    
elif STRUCTURE == 3:
    var_names = (['b_Tm_E_Tm', 'b_Tm_E_Tp', 'b_Tm_F_Tm', 'b_Tm_F_Tp', 'b_Tm_H_Tm', 'b_Tm_H_Tp', 'b_Tm_Tp_E', 'b_Tm_Tp_F', 'b_Tm_Tp_H', 'b_Tp_E_Tm', 'b_Tp_E_Tp', 'b_Tp_F_Tm', 'b_Tp_F_Tp', 'b_Tp_H_Tm', 'b_Tp_H_Tp'])

elif STRUCTURE == 4:
    var_names = (['b', 'b_Tm_E_Tm', 'b_Tm_E_Tp', 'b_Tm_F_Tm', 'b_Tm_F_Tp', 'b_Tm_H_Tm', 'b_Tm_H_Tp', 'b_Tm_Tp_E', 'b_Tm_Tp_F', 'b_Tm_Tp_H', 'b_Tp_E_Tm', 'b_Tp_E_Tp', 'b_Tp_F_Tm', 'b_Tp_F_Tp', 'b_Tp_H_Tm', 'b_Tp_H_Tp'])
# Build the polynomial ring over GF(3) in all variables with lex order
R = PolynomialRing(F3, var_names, order='lex')

# Create a dictionary for easy variable access by name
var = {name: R(name) for name in var_names}

print(f"Polynomial ring: {R}")
print(f"Number of variables: {len(var_names)}")
print(f"Variables: {var_names}")

Polynomial ring: Multivariate Polynomial Ring in b_Tm_E_Tm, b_Tm_E_Tp, b_Tm_F_Tm, b_Tm_F_Tp, b_Tm_H_Tm, b_Tm_H_Tp, b_Tm_Tp_E, b_Tm_Tp_F, b_Tm_Tp_H, b_Tp_E_Tm, b_Tp_E_Tp, b_Tp_F_Tm, b_Tp_F_Tp, b_Tp_H_Tm, b_Tp_H_Tp over Finite Field of size 3
Number of variables: 15
Variables: ['b_Tm_E_Tm', 'b_Tm_E_Tp', 'b_Tm_F_Tm', 'b_Tm_F_Tp', 'b_Tm_H_Tm', 'b_Tm_H_Tp', 'b_Tm_Tp_E', 'b_Tm_Tp_F', 'b_Tm_Tp_H', 'b_Tp_E_Tm', 'b_Tp_E_Tp', 'b_Tp_F_Tm', 'b_Tp_F_Tp', 'b_Tp_H_Tm', 'b_Tp_H_Tp']


---
## Part 2 — Paste Your Polynomial System

### How to extract the polynomials from your SymPy code

In `linear_sys.py`, after the variable `reduced_polys` is built (line ~693),
add the following export block:

```python
# === Export reduced_polys for Sage ===
# Run this AFTER the Groebner section, with the correct structure_version set

import os

# Get all free symbols across all polynomials
all_syms = set()
for p in reduced_polys_mod3:  # use reduced_polys_mod3 (already reduced mod 3)
    all_syms |= p.free_symbols

with open('polys_for_sage.py', 'w') as f:
    f.write("# Auto-generated polynomial system for SageMath verification\n")
    f.write(f"# Structure version: {structure_version}\n\n")
    f.write("polys_raw = [\n")
    for p in reduced_polys_mod3:
        f.write(f"    '{sp.srepr(p)}',\n")
    f.write("]\n")
print('Exported polys_for_sage.py')
```

Then load and parse those in the cell below.

**Alternatively**, you can directly copy-paste the polynomial strings from the
exported `.txt` file (e.g. `polynomials_in_b_..._lss_1.txt`) and rewrite
them as Sage expressions in the cell below.

In [ ]:
# ============================================================
# OPTION A: Paste polynomial strings directly
# Replace the examples below with your actual polynomials.
# Each string is a polynomial in the variables defined in var_names.
# Sage will parse them as elements of R automatically.
# ============================================================

# Example format (replace with your actual polynomials from the .txt export):
poly_strings = [
    # Paste your actual polynomial strings here, one per line, as strings.
    # Example (NOT your actual polynomials):
    # 'tau0*tau1 + u*tau2 - v',
    # 'tau0^2 + tau3*v - 1',
    # ...
    
    # PLACEHOLDER — replace entirely with your system
    'tau0',   
]

# For Structure 2: also add the invertibility relation g * ginv - 1 = 0
if STRUCTURE == 2:
    poly_strings.append('g*ginv - 1')
    print("Added invertibility relation: g*ginv - 1 = 0")

# Parse all strings into ring elements
polys_sage = [R(p) for p in poly_strings]

print(f"\nNumber of polynomials loaded: {len(polys_sage)}")
print("\nFirst 5 polynomials (as Sage ring elements):")
for p in polys_sage[:5]:
    print(" ", p)

In [56]:
# ============================================================
# OPTION B: Load from the exported polys_for_sage.py file
# (Use this if you ran the export block above in SymPy)
# ============================================================

# Uncomment and adjust the path if using this option:

import importlib.util, sys
spec = importlib.util.spec_from_file_location("polys_raw", "/Users/josemarialoza/capstone/polys_files_sage/polys_for_sage_lss3.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

poly_strings = mod.polys_raw

if STRUCTURE == 2:
    poly_strings.append('g*ginv - 1')
    
if STRUCTURE == 4:
    poly_strings.append('b**2 + 1')
    
# Sanity check: print first 3 raw strings before parsing
print("Sample raw strings from file:")
for s in poly_strings[:3]:
    print(" ", s)
    
polys_sage = [R(p) for p in poly_strings]

print(f"\nLoaded {len(polys_sage)} polynomials into ring {R.base_ring()}")
print("\nFirst 3 as Sage ring elements:")
for p in polys_sage[:3]:
    print(" ", p)

Sample raw strings from file:
  -b_Tm_E_Tp*b_Tp_F_Tm + b_Tm_E_Tp + b_Tm_F_Tp*b_Tp_E_Tm + b_Tp_F_Tm + b_Tp_H_Tp
  -b_Tm_E_Tm*b_Tp_F_Tm + b_Tm_E_Tm + b_Tm_F_Tm*b_Tp_E_Tm - b_Tp_E_Tm*b_Tp_F_Tp + b_Tp_E_Tp*b_Tp_F_Tm - b_Tp_E_Tp + b_Tp_H_Tm
  b_Tm_E_Tm*b_Tm_F_Tp - b_Tm_E_Tp*b_Tm_F_Tm + b_Tm_E_Tp*b_Tp_F_Tp + b_Tm_F_Tm - b_Tm_F_Tp*b_Tp_E_Tp + b_Tm_H_Tp - b_Tp_F_Tp

Loaded 203 polynomials into ring Finite Field of size 3

First 3 as Sage ring elements:
  -b_Tm_E_Tp*b_Tp_F_Tm + b_Tm_E_Tp + b_Tm_F_Tp*b_Tp_E_Tm + b_Tp_F_Tm + b_Tp_H_Tp
  -b_Tm_E_Tm*b_Tp_F_Tm + b_Tm_E_Tm + b_Tm_F_Tm*b_Tp_E_Tm - b_Tp_E_Tm*b_Tp_F_Tp + b_Tp_E_Tp*b_Tp_F_Tm - b_Tp_E_Tp + b_Tp_H_Tm
  b_Tm_E_Tm*b_Tm_F_Tp - b_Tm_E_Tp*b_Tm_F_Tm + b_Tm_E_Tp*b_Tp_F_Tp + b_Tm_F_Tm - b_Tm_F_Tp*b_Tp_E_Tp + b_Tm_H_Tp - b_Tp_F_Tp


---
## Part 3 — Compute the Gröbner Basis

SageMath uses Singular as its backend for Gröbner basis computations,
which is significantly more reliable than SymPy's Buchberger implementation
for large systems.

In [57]:
# Build the ideal
I = R.ideal(polys_sage)
print("Ideal constructed.")
print(f"Number of generators: {len(polys_sage)}")

Ideal constructed.
Number of generators: 203


In [58]:
# Compute the reduced Gröbner basis (lex order, as specified in ring definition)
# This may take a few minutes for large systems — Sage will show progress
print("Computing Gröbner basis over GF(3)...")
GB = I.groebner_basis()

print(f"\nGröbner basis computed.")
print(f"Basis size: {len(GB)}")
print(f"Basis elements:")
for g in GB:
    print(" ", g)

Computing Gröbner basis over GF(3)...

Gröbner basis computed.
Basis size: 1
Basis elements:
  1


In [59]:
# ============================================================
# Primary check: is 1 in the Gröbner basis?
# ============================================================
one = R(1)
is_inconsistent = (GB == [one]) or (one in GB)

print("=" * 55)
if is_inconsistent:
    print("RESULT: Gröbner basis = {1}")
    print()
    print("The ideal I equals the whole ring R.")
    print("The system of polynomial equations is INCONSISTENT.")
    print()
    print("Interpretation: No solution exists in K^n for any")
    print("field K of characteristic 3. The graded left-symmetric")
    print(f"identity cannot be satisfied for Structure {STRUCTURE}.")
else:
    print("RESULT: Gröbner basis ≠ {1}")
    print()
    print("The system may be consistent. Inspect the basis above.")
print("=" * 55)

RESULT: Gröbner basis = {1}

The ideal I equals the whole ring R.
The system of polynomial equations is INCONSISTENT.

Interpretation: No solution exists in K^n for any
field K of characteristic 3. The graded left-symmetric
identity cannot be satisfied for Structure 3.


---
## Part 4 — Extract the Polynomial Certificate

If the Gröbner basis is {1}, we can explicitly extract the polynomials h₁,...,hₖ
such that h₁f₁ + ... + hₖfₖ = 1. This is the **Nullstellensatz certificate** —
the explicit proof that the ideal contains 1.

This certificate is what you cite in your paper to justify generalization to
any field of characteristic 3.

In [60]:
if is_inconsistent:
    print("Computing Nullstellensatz certificate: h_i such that Σ h_i * f_i = 1")
    print("(This may take a moment for large systems)\n")
    
    try:
        # lift() computes the h_i coefficients
        # one.lift(I) returns [h_1, ..., h_k] such that sum(h_i * f_i) = 1
        certificate = one.lift(I)
        
        print(f"Certificate found: {len(certificate)} multiplier polynomials h_i")
        print()
        
        # Verify the certificate
        check = sum(h * f for h, f in zip(certificate, polys_sage))
        print(f"Verification — Σ h_i * f_i = {check}  (should be 1)")
        assert check == one, "Certificate verification FAILED!"
        print("Certificate verification PASSED ✓")
        print()
        
        # Print nonzero multipliers with their corresponding polynomial
        print("Nonzero multipliers (h_i, f_i pairs):")
        for i, (h, f) in enumerate(zip(certificate, polys_sage)):
            if h != R(0):
                print(f"\n  h_{i} = {h}")
                print(f"  f_{i} = {f}")
                
        # Export nullstellensatz certificate to text file
        export_path = f'/Users/josemarialoza/capstone/nullstellensatz_certs/null_cert_structure_{STRUCTURE}.txt'
        with open(export_path, "w") as cert_file:
            cert_file.write(f"Nullstellensatz Certificate for Structure {STRUCTURE}\n")
            cert_file.write("=" * 50 + "\n\n")
            cert_file.write(f"Certificate polynomials h_i (nonzero):\n\n")
            for i, (h, f) in enumerate(zip(certificate, polys_sage)):
                if h != R(0):
                    cert_file.write(f"h_{i} = {h}\n")
                    cert_file.write(f"f_{i} = {f}\n\n")
            cert_file.write(f"Verification: Σ h_i * f_i = {check} (should be 1)\n")
                
    except Exception as e:
        print(f"Certificate extraction failed: {e}")
        print("The basis {1} still confirms inconsistency; the certificate is implicit.")
else:
    print("System is consistent — no certificate needed.")

Computing Nullstellensatz certificate: h_i such that Σ h_i * f_i = 1
(This may take a moment for large systems)

Certificate found: 203 multiplier polynomials h_i

Verification — Σ h_i * f_i = 1  (should be 1)
Certificate verification PASSED ✓

Nonzero multipliers (h_i, f_i pairs):

  h_71 = 1
  f_71 = b_Tm_F_Tm - b_Tm_F_Tp*b_Tp_H_Tm + b_Tm_H_Tp*b_Tp_F_Tm - b_Tm_H_Tp

  h_112 = -b_Tp_F_Tm*b_Tp_F_Tp + b_Tp_F_Tp + 1
  f_112 = b_Tm_F_Tp*b_Tp_H_Tm + b_Tm_H_Tp - b_Tp_E_Tp + b_Tp_F_Tp*b_Tp_H_Tp + b_Tp_F_Tp

  h_113 = 1
  f_113 = b_Tm_F_Tm*b_Tp_H_Tm + b_Tm_H_Tm - b_Tp_E_Tm + b_Tp_F_Tm*b_Tp_H_Tp + b_Tp_F_Tm - b_Tp_H_Tp

  h_120 = b_Tm_H_Tp*b_Tp_F_Tp^2 + b_Tm_H_Tp*b_Tp_F_Tp + b_Tm_H_Tp - b_Tp_E_Tp*b_Tp_F_Tp^2 - b_Tp_E_Tp*b_Tp_F_Tp - b_Tp_E_Tp - b_Tp_F_Tp^3*b_Tp_H_Tp^2 - b_Tp_F_Tp^2*b_Tp_H_Tp - b_Tp_F_Tp*b_Tp_H_Tp + b_Tp_F_Tp + b_Tp_H_Tp^2 - 1
  f_120 = b_Tm_H_Tp*b_Tp_E_Tm + b_Tp_E_Tp*b_Tp_H_Tp - b_Tp_E_Tp

  h_122 = b_Tm_H_Tp*b_Tp_F_Tp^2 - b_Tm_H_Tp*b_Tp_F_Tp + b_Tm_H_Tp - b_Tp_E_Tp*b_Tp_F_Tp^2

---
## Part 5 — Cross-check: Ideal Membership Test

An independent check: test whether 1 is in the ideal directly,
without relying on the Gröbner basis output.

In [ ]:
# Direct ideal membership test
one_in_ideal = one in I

print(f"Direct membership test: 1 ∈ I? {one_in_ideal}")

# Cross-check consistency between the two methods
if one_in_ideal == is_inconsistent:
    print("Both methods agree ✓")
else:
    print("WARNING: Methods disagree — investigate further.")

---
## Part 6 — (Optional) Verify Over a Larger Characteristic-3 Field

As a sanity check, we can verify that the ideal also contains 1 when the
coefficient field is extended to GF(9) = GF(3²) or GF(27) = GF(3³).

By the Lemma (F_3 ⊆ GF(3^k)), the same polynomials generate the ideal
over the larger field, and so if 1 ∈ I over GF(3), it remains in I over GF(9), etc.
This cell makes that concrete and explicit.

In [ ]:
for q in [9, 27]:  # GF(3^2) and GF(3^3)
    try:
        Fq = GF(q)  # Extension field of GF(3)
        Rq = PolynomialRing(Fq, var_names, order='lex')
        
        # Rewrite polynomials over the extension field
        # Coefficients are in GF(3) ⊂ GF(q), so this is just a reinterpretation
        polys_q = [Rq(str(p)) for p in polys_sage]
        
        if STRUCTURE == 2:
            polys_q.append(Rq('g*ginv - 1'))
        
        I_q = Rq.ideal(polys_q)
        GB_q = I_q.groebner_basis()
        
        one_q = Rq(1)
        result = (GB_q == [one_q]) or (one_q in GB_q)
        print(f"GF({q}): Gröbner basis = {{1}}? {result}")
        
    except Exception as e:
        print(f"GF({q}): Failed — {e}")

---
## Part 7 — (Optional) Verify Both Structures in One Pass

In [ ]:
# Summary table
print("=" * 55)
print("SUMMARY")
print("=" * 55)
print(f"Structure version:         {STRUCTURE}")
print(f"Base field:                GF(3)")
print(f"Number of polynomials:     {len(polys_sage)}")
print(f"Number of variables:       {len(var_names)}")
print(f"Gröbner basis = {{1}}:      {is_inconsistent}")
print(f"1 ∈ I (direct check):      {one_in_ideal}")
print()
if is_inconsistent:
    print("CONCLUSION: The graded left-symmetric identity system")
    print(f"for Structure {STRUCTURE} of Burde is inconsistent over any")
    print("field K of characteristic 3. osp(1|2) does not admit")
    print(f"a left-symmetric structure extending Burde's Structure {STRUCTURE}.")
print("=" * 55)

---
## Appendix — Theoretical Justification (for Paper Reference)

**Why computing over GF(3) with parameters as indeterminates suffices for all char-3 fields:**

Let $f_1, \ldots, f_k \in \mathbb{F}_3[\tau_1, \ldots, \tau_n, u, v]$ be the polynomials
encoding the graded left-symmetric identity constraints, where $u, v$ (or $g$) are
Burde's free parameters treated as indeterminates.

**Step 1 (Computation).** The Gröbner basis of $I = \langle f_1, \ldots, f_k \rangle$ in
$\mathbb{F}_3[\tau, u, v]$ is $\{1\}$, so there exist explicit polynomials
$h_1, \ldots, h_k \in \mathbb{F}_3[\tau, u, v]$ (the Nullstellensatz certificate above) such that:
$$\sum_i h_i \cdot f_i = 1 \quad \text{in } \mathbb{F}_3[\tau_1,\ldots,\tau_n, u, v] \tag{★}$$

**Step 2 (Extension Lemma).** For any field $\mathbb{K}$ with $\mathrm{char}(\mathbb{K}) = 3$,
there is a canonical embedding $\iota: \mathbb{F}_3 \hookrightarrow \mathbb{K}$.
Since $f_i, h_i \in \mathbb{F}_3[\tau, u, v] \subseteq \mathbb{K}[\tau, u, v]$,
equation (★) holds as an identity in $\mathbb{K}[\tau, u, v]$ as well —
it is the same equation, now viewed inside a larger ring.

**Step 3 (No solutions).** By the Universal Property of polynomial rings, for any
$(a_1,\ldots,a_n,\alpha,\beta) \in \mathbb{K}^{n+2}$, substituting into (★) gives
$\sum_i h_i(a,\alpha,\beta) \cdot f_i(a,\alpha,\beta) = 1$ in $\mathbb{K}$.
If $(a, \alpha, \beta)$ were a solution, every $f_i$ would vanish and the left side
would be $0$, contradicting $1 \neq 0$ in $\mathbb{K}$. $\square$